# Đánh giá cuối — Student Dropout

Notebook huấn luyện lại các cấu hình đã chốt cho bốn mô hình trên
hai bộ đặc trưng A và A+B. Các cấu hình được chọn dựa trên CV Macro-F1
của tập development.

Sau khi chốt cấu hình, mỗi pipeline được huấn luyện trên toàn bộ
development và đánh giá trên cùng tập test giữ riêng. Notebook xuất
bảng kết quả và lưu pipeline của mô hình được chọn cho ứng dụng demo.

## 1. Xác định đường dẫn dữ liệu

Tìm thư mục dự án và kiểm tra vị trí của `development.csv` cùng
các file CSV trong thư mục `data`.

In [1]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()

project_dir = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if folder.name == "ML01_Student_Dropout"
    ),
    None,
)

if project_dir is None:
    raise FileNotFoundError(
        "Mở thư mục ML01_Student_Dropout trong VS Code rồi chạy lại cell này."
    )

data_dir = project_dir / "data"
development_files = sorted(data_dir.rglob("development.csv"))

if len(development_files) != 1:
    raise ValueError(
        f"Cần đúng một file development.csv trong {data_dir}. "
        f"Tìm thấy {len(development_files)} file: {development_files}"
    )

development_path = development_files[0]

print("Python:", sys.executable)
print("Thư mục dự án:", project_dir)
print("File development:", development_path.relative_to(project_dir))
print("\nCác file CSV trong data:")

for csv_path in sorted(data_dir.rglob("*.csv")):
    print("-", csv_path.relative_to(project_dir))

Python: c:\Users\Admin\Documents\GitHub\sgu-MAYHOC\.venv\Scripts\python.exe
Thư mục dự án: C:\Users\Admin\Documents\GitHub\sgu-MAYHOC\ML01_Student_Dropout
File development: data\processed\development.csv

Các file CSV trong data:
- data\processed\development.csv
- data\processed\test.csv
- data\raw\student_dropout.csv


## 2. Chuẩn bị đặc trưng A và A+B

Dùng lại hàm `load_development` đã dùng trong thực nghiệm
Tập A gồm 36 đặc trưng gốc. Tập A+B bổ sung tỷ lệ môn đạt ở học kỳ 1 và cờ cho biết sinh viên đăng ký 0 môn ở học kỳ 1.

In [2]:
feature_b_dir = project_dir / "feature_b"

if not (feature_b_dir / "common.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy {feature_b_dir / 'common.py'}")

if str(feature_b_dir) not in sys.path:
    sys.path.insert(0, str(feature_b_dir))

from common import load_development

X_dev_a, X_dev_ab, y_dev = load_development(development_path)

print("\nKích thước dữ liệu development")
print("A:", X_dev_a.shape)
print("A+B:", X_dev_ab.shape)
print("Nhãn:", y_dev.shape)

added_columns = [col for col in X_dev_ab.columns if col not in X_dev_a.columns]
print("Hai cột bổ sung:", added_columns)

3539 mẫu | A: 36 cột | A+B: 38 cột
Số mẫu đăng ký 0 môn HK1: 148

Kích thước dữ liệu development
A: (3539, 36)
A+B: (3539, 38)
Nhãn: (3539,)
Hai cột bổ sung: ['hk1_pass_rate', 'hk1_zero_enrolled']


## 3. Cấu hình đã chọn từ thực nghiệm

Với mỗi mô hình, chọn một cấu hình trên A và một cấu hình trên A+B dựa trên CV Macro-F1 của tập development. Xét cả cấu hình mốc và các cấu hình thử có cùng quy trình đánh giá.

Ghi đầy đủ tham số và điểm CV từ lượt chạy được chọn trước khi đánh giá trên tập test.

In [3]:
import pandas as pd

lr = dict(C=0.1, class_weight="balanced", solver="lbfgs", max_iter=3000)
rf = dict(
    n_estimators=200, max_depth=None, min_samples_leaf=1,
    class_weight="balanced", random_state=42, n_jobs=1,
)
svm = dict(kernel="rbf", C=5, gamma=0.01, class_weight="balanced")
xgb = dict(
    n_estimators=150, max_depth=6, learning_rate=0.1,
    objective="multi:softprob", num_class=3, eval_metric="mlogloss",
    tree_method="hist", random_state=42, n_jobs=1,
)

selected_configs = pd.DataFrame(
    [
        ("A", "Logistic Regression", lr, 0.717353),
        ("A", "Random Forest", {**rf, "n_estimators": 300}, 0.722722),
        ("A", "SVM", svm, 0.723261),
        ("A", "XGBoost", xgb, 0.713649),
        ("A+B", "Logistic Regression", {**lr, "C": 0.2}, 0.718329),
        ("A+B", "Random Forest", rf, 0.723427),
        ("A+B", "SVM", {**svm, "C": 10, "gamma": 0.005}, 0.723082),
        ("A+B", "XGBoost", {**xgb, "max_depth": 4}, 0.717022),
    ],
    columns=["feature_set", "model", "params", "cv_macro_f1"],
)

display(selected_configs)

,feature_set,model,params,cv_macro_f1
0,A,Logistic Regression,"{'C': 0.1, 'class_weight': 'balanced', 'solver...",0.717353
1,A,Random Forest,"{'n_estimators': 300, 'max_depth': None, 'min_...",0.722722
2,A,SVM,"{'kernel': 'rbf', 'C': 5, 'gamma': 0.01, 'clas...",0.723261
3,A,XGBoost,"{'n_estimators': 150, 'max_depth': 6, 'learnin...",0.713649
4,A+B,Logistic Regression,"{'C': 0.2, 'class_weight': 'balanced', 'solver...",0.718329
5,A+B,Random Forest,"{'n_estimators': 200, 'max_depth': None, 'min_...",0.723427
6,A+B,SVM,"{'kernel': 'rbf', 'C': 10, 'gamma': 0.005, 'cl...",0.723082
7,A+B,XGBoost,"{'n_estimators': 150, 'max_depth': 4, 'learnin...",0.717022


## 4. Huấn luyện lại và đánh giá trên tập test

Với mỗi cấu hình đã chốt, huấn luyện pipeline trên toàn bộ development và dự đoán trên tập test giữ riêng. Báo cáo Macro-F1 và Recall lớp Dropout trên test cho cả hai bộ đặc trưng A và A+B.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score
from sklearn.svm import SVC
from xgboost import XGBClassifier

from common import LABELS, make_pipeline

model_classes = {
    "Logistic Regression": LogisticRegression,
    "Random Forest": RandomForestClassifier,
    "SVM": SVC,
    "XGBoost": XGBClassifier,
}
scaled_models = {"Logistic Regression", "SVM"}
development_features = {"A": X_dev_a, "A+B": X_dev_ab}

final_results = None
trained_pipelines = {}
if selected_configs[["params", "cv_macro_f1"]].isna().any().any():
    print("Chưa đủ 8 cấu hình. Điền các ô None ở Cell 7 rồi chạy lại.")
else:
    test_path = development_path.with_name("test.csv")
    X_test_a, X_test_ab, y_test = load_development(test_path)
    test_features = {"A": X_test_a, "A+B": X_test_ab}
    dropout_label = LABELS.index("Dropout")
    results = []

    for config in selected_configs.itertuples(index=False):
        feature_set = config.feature_set
        model = model_classes[config.model](**config.params)
        pipeline = make_pipeline(
            model,
            scale_numeric=config.model in scaled_models,
            add_b=feature_set == "A+B",
        )

        pipeline.fit(development_features[feature_set], y_dev)
        trained_pipelines[(feature_set, config.model)] = pipeline
        predictions = pipeline.predict(test_features[feature_set])

        results.append({
            "feature_set": feature_set,
            "model": config.model,
            "cv_macro_f1": config.cv_macro_f1,
            "test_macro_f1": f1_score(
                y_test, predictions, average="macro", zero_division=0
            ),
            "test_recall_dropout": recall_score(
                y_test, predictions, labels=[dropout_label],
                average=None, zero_division=0,
            )[0],
        })

    final_results = pd.DataFrame(results)
    display(final_results.round(6))

885 mẫu | A: 36 cột | A+B: 38 cột
Số mẫu đăng ký 0 môn HK1: 32


,feature_set,model,cv_macro_f1,test_macro_f1,test_recall_dropout
0,A,Logistic Regression,0.717353,0.700098,0.686620
1,A,Random Forest,0.722722,0.714487,0.714789
2,A,SVM,0.723261,0.694739,0.683099
3,A,XGBoost,0.713649,0.694288,0.711268
4,A+B,Logistic Regression,0.718329,0.697171,0.683099
5,A+B,Random Forest,0.723427,0.707345,0.725352
6,A+B,SVM,0.723082,0.698823,0.683099
7,A+B,XGBoost,0.717022,0.704011,0.728873


In [5]:
import joblib

if final_results is None:
    raise RuntimeError("Điền đủ cấu hình rồi chạy lại mục 4.")

winner = selected_configs.loc[selected_configs["cv_macro_f1"].idxmax()]
model_key = (winner["feature_set"], winner["model"])

output_path = project_dir / "final_test_results.csv"
final_results.to_csv(output_path, index=False, encoding="utf-8-sig")

model_path = project_dir / "models" / "final_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "pipeline": trained_pipelines[model_key],
    "feature_set": winner["feature_set"],
    "model_name": winner["model"],
    "labels": LABELS,
}, model_path)

print(f"Đã lưu bảng điểm: {output_path}")
print(f"Đã lưu pipeline: {winner['model']} trên {winner['feature_set']}")

Đã lưu bảng điểm: C:\Users\Admin\Documents\GitHub\sgu-MAYHOC\ML01_Student_Dropout\final_test_results.csv
Đã lưu pipeline: Random Forest trên A+B
